In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q08/q08_rewrite/checkpoints/pre_cell_7.pickle

In [ ]:
%%cudf.pandas.profile
### cell 7 ###

# cell 7 optimized
# 1) push all filters and selects to GPU-friendly operations
part_filtered = (
    part[part.P_TYPE == "ECONOMY ANODIZED STEEL"][["P_PARTKEY"]]
)

lineitem_filtered = lineitem.assign(
    VOLUME=lineitem.L_EXTENDEDPRICE * (1.0 - lineitem.L_DISCOUNT)
)[["L_PARTKEY", "L_SUPPKEY", "L_ORDERKEY", "VOLUME"]]

supplier_filtered = supplier[["S_SUPPKEY", "S_NATIONKEY"]]

# use string literals for date filters to avoid pd.Timestamp on CPU
orders_filtered = (
    orders[
        (orders.O_ORDERDATE >= "1995-01-01")
        & (orders.O_ORDERDATE < "1997-01-01")
    ]
    .assign(O_YEAR=orders.O_ORDERDATE.dt.year)
    [["O_ORDERKEY", "O_CUSTKEY", "O_YEAR"]]
)

customer_filtered = customer[["C_CUSTKEY", "C_NATIONKEY"]]

n1_filtered = nation[["N_NATIONKEY", "N_REGIONKEY"]]

n2_filtered = (
    nation[["N_NATIONKEY", "N_NAME"]]
    .rename(columns={"N_NAME": "NATION"})
)

region_filtered = region[region.R_NAME == "AMERICA"][["R_REGIONKEY"]]

# chain all merges in one pipeline
total = (
    part_filtered
    .merge(lineitem_filtered, left_on="P_PARTKEY", right_on="L_PARTKEY")
    .merge(supplier_filtered,   left_on="L_SUPPKEY", right_on="S_SUPPKEY")
    .merge(orders_filtered,     left_on="L_ORDERKEY", right_on="O_ORDERKEY")
    .merge(customer_filtered,   left_on="O_CUSTKEY", right_on="C_CUSTKEY")
    .merge(n1_filtered,        left_on="C_NATIONKEY", right_on="N_NATIONKEY")
    .merge(n2_filtered,        left_on="S_NATIONKEY", right_on="N_NATIONKEY")
    .merge(region_filtered,    left_on="N_REGIONKEY", right_on="R_REGIONKEY")
    [["VOLUME", "O_YEAR", "NATION"]]
)

# 2) compute market share without a Python UDF
# add a GPU‐computed column for Brazil volume
total["BRAZIL_VOL"] = total.VOLUME.where(total.NATION == "BRAZIL", 0)

# groupby on GPU and sum both columns
agg = (
    total
    .groupby("O_YEAR")[["VOLUME", "BRAZIL_VOL"]]
    .sum()
    .reset_index()
)

# vectorized division for market share
agg["MKT_SHARE"] = agg.BRAZIL_VOL / agg.VOLUME

# select and sort final result
total = agg[["O_YEAR", "MKT_SHARE"]].sort_values("O_YEAR")